In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Neural network weights example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [2]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

# Преглед на данните
df.show(5)

+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|Gender|Age|   Degree|PlatformUsage|PlatformHours|OutPlatformHours|Satisfaction|Self-assessment|PreferredFormat|
+------+---+---------+-------------+-------------+----------------+------------+---------------+---------------+
|     Ж| 21|Бакалавър|           Да|            6|               2|           4|              4|          Видео|
|     М| 23| Магистър|           Да|            8|               1|           5|              5|          Видео|
|     Ж| 20|Бакалавър|           Не|            0|               3|           3|              3|          Текст|
|     Ж| 22|Бакалавър|           Да|            5|               1|           4|              4|         Смесен|
|     М| 24| Магистър|           Да|            9|               0|           5|              5|          Видео|
+------+---+---------+-------------+-------------+----------------+------------+---------------+

In [3]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Подготовка на данните
feature_cols = ["Age","PlatformHours","OutPlatformHours","Self-assessment"]
# Обединяване на всички характеристики във features колона
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df)

# Разделяне на данните
train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

# Архитектура на мрежата
layers = [4, 7, 8, 6]
# Създаване на модела
mlp = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="Satisfaction",
    layers=layers, maxIter=100)
# Обучение
model = mlp.fit(train_data)

# Прогнозиране
predictions = model.transform(test_data)
# Оценка
evaluator = MulticlassClassificationEvaluator(
    labelCol="Satisfaction", predictionCol="prediction",
    metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 0.5


In [6]:
import numpy as np

# Извличане на теглата
weights = model.weights.toArray()
# Средна абсолютна стойност
mean_abs = np.mean(np.abs(weights))

print("Тегла:")
print(weights)
print("Средна абсолютна стойност:")
print(mean_abs)

Тегла:
[ 3.48714448e+00 -6.34009161e-01 -1.81192888e+01  9.85240150e-01
 -2.54782432e+00 -1.20937474e+01 -1.42285313e+00 -8.19282637e+00
  2.88152446e-01 -5.16498812e-01  4.13547802e-01  3.73129272e-01
  4.91912869e+01  1.53270466e+00  4.80807308e+00 -6.52251467e-01
 -2.61807933e+00 -1.14097722e+00  9.55383771e-01 -1.68540314e+01
  5.74790109e+00 -2.55998069e+00 -7.89880851e-01 -3.17895584e+00
 -7.13409994e-01 -3.57408732e-01  9.62638016e+00  5.78445813e+00
  7.99783515e-02 -7.77498054e-01 -1.27256408e+00  3.78169450e-01
 -7.09782176e-01 -1.59245408e+00  9.72770992e-02 -7.25565482e+00
 -2.12456228e+01 -1.08756110e+01  2.49243833e+00  2.58342661e+01
  3.29893434e+00  3.22163085e+01  1.46170298e+01 -4.97529217e-01
  6.08435973e-01  5.97740530e-01  1.06895199e-01 -2.44028025e-01
  1.44794583e-01  2.60170329e-01 -2.15820574e-01 -2.28570381e+00
 -3.43618435e+00 -1.02753069e+00  1.51875925e+00  5.69663604e-01
  1.38928649e+00  5.00549702e-01 -1.02355606e+00 -5.53836930e+00
 -2.17503160e+01 -